# Import Libraries

In [1]:
import pandas as pd
import numpy as np
import spacy
import os
from tqdm import tqdm 
from sklearn.metrics import classification_report
import os
import requests
from openai import OpenAI, RateLimitError
import time
import random
from google import genai

import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_colwidth', None)  # Show full content in each cell
pd.set_option('display.width', 1000)  # Set max width

# Load spaCy's English model
nlp = spacy.load('en_core_web_sm')

# Pre-Processing

In [2]:
label_mapper = {
    'knowledge' : 0,
    'comprehension' : 1,
    'application' : 2,
    'analysis' : 3,
    'synthesis' : 4,
    'evaluation' : 5
}

mapping = {
    'knowledge': 'knowledge',
    'remember': 'knowledge',
    'comprehension': 'comprehension',
    'understand': 'comprehension',
    'application': 'application',
    'apply': 'application',
    'analysis': 'analysis',
    'analyse': 'analysis',
    'evaluation': 'evaluation',
    'evaluate': 'evaluation',
    'synthesis': 'synthesis',
    'create': 'synthesis'
}

q_df = pd.read_csv(os.getcwd().replace('notebook' , 'dataset') + '/dataset4.csv')
queries = q_df['question']
q_df['label'] = q_df['label'].str.lower()
q_df['label'] = q_df['label'].replace(mapping)
label = q_df['label'].str.lower().map(label_mapper)
print(q_df['label'].value_counts())

label
synthesis        29
knowledge        22
evaluation       21
comprehension    20
analysis         19
application      15
Name: count, dtype: int64


# API Setup

In [3]:
# Sonar
 
api_key = os.environ.get("PERPLEXITY_API_KEY")

if api_key:
    print('successful')

url = "https://api.perplexity.ai/chat/completions"
headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

successful


In [4]:
api_key = os.environ.get("GOOGLE_API_KEY")

if api_key:
    print('successful')

google_client = genai.Client(api_key=api_key)

successful


In [7]:
# Groq

api_key = os.environ.get("GROQ_API_KEY")

if api_key:
    print('successful')

groq_client = OpenAI(
    base_url = "https://api.groq.com/openai/v1",
    api_key = api_key
)

successful


# Zero-Shot with Context

## GPT-OSS-120B

In [8]:
zso_pred_labels = []

for query in tqdm(queries):
    chat_completion = groq_client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": f"""Given the query below, classify which Bloom's Taxonomy level it belongs to.
                    Levels: [knowledge, comprehension, application, analysis, synthesis, evaluation]
                
                query : {query}""",
            }
        ],
        model="openai/gpt-oss-120b",
    )

    reply = chat_completion.choices[0].message.content.lower()

    while reply not in ['knowledge', 'comprehension', 'application', 'analysis', 'synthesis', 'evaluation']:
        chat_completion = groq_client.chat.completions.create(
            messages=[
                {
                    "role": "user",
                    "content": f"""Extract the blooms level from previous reponse. Answer only in one word without punctuation from: 
                        [knowledge , comprehension , application , analysis, synthesis , evaluation]
                    previous response : {reply}""",
                }
            ],
            model="openai/gpt-oss-20b",
        )

        reply = chat_completion.choices[0].message.content.lower()

    zso_pred_labels.append(reply.lower())

100%|██████████| 126/126 [04:48<00:00,  2.29s/it]


In [9]:
print(classification_report(label , [label_mapper[key.lower()] for key in zso_pred_labels]))

              precision    recall  f1-score   support

           0       0.88      0.95      0.91        22
           1       0.87      0.65      0.74        20
           2       0.45      0.33      0.38        15
           3       0.81      0.68      0.74        19
           4       0.65      0.90      0.75        29
           5       0.90      0.86      0.88        21

    accuracy                           0.76       126
   macro avg       0.76      0.73      0.74       126
weighted avg       0.77      0.76      0.75       126



## LLAMA4-Scout

In [10]:
zsl_pred_labels = []

for query in tqdm(queries):
    chat_completion = groq_client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": f"""Given the query below, classify which Bloom's Taxonomy level it belongs to.
                    Levels: [knowledge, comprehension, application, analysis, synthesis, evaluation]
                
                query : {query}""",
            }
        ],
        model="meta-llama/llama-4-scout-17b-16e-instruct",
    )

    reply = chat_completion.choices[0].message.content.lower()

    while reply not in ['knowledge', 'comprehension', 'application', 'analysis', 'synthesis', 'evaluation']:
        chat_completion = groq_client.chat.completions.create(
            messages=[
                {
                    "role": "user",
                    "content": f"""Extract the blooms level from previous reponse. Answer only in one word without punctuation from: 
                        [knowledge , comprehension , application , analysis, synthesis , evaluation]
                    previous response : {reply}""",
                }
            ],
            model="meta-llama/llama-4-scout-17b-16e-instruct",
        )

        reply = chat_completion.choices[0].message.content.lower()

    zsl_pred_labels.append(reply.lower())

100%|██████████| 126/126 [09:21<00:00,  4.46s/it]


In [11]:
print(classification_report(label , [label_mapper[key.lower()] for key in zsl_pred_labels]))

              precision    recall  f1-score   support

           0       0.81      0.95      0.88        22
           1       0.75      0.60      0.67        20
           2       0.29      0.27      0.28        15
           3       0.80      0.63      0.71        19
           4       0.69      0.86      0.77        29
           5       0.84      0.76      0.80        21

    accuracy                           0.71       126
   macro avg       0.70      0.68      0.68       126
weighted avg       0.71      0.71      0.71       126



## GEMINI

In [ ]:
zsg_pred_labels = []

for query in tqdm(queries):
    response = google_client.models.generate_content(
        model="gemini-2.5-flash", 
        contents=
            f"""Given the query below, classify which Bloom's Taxonomy level it belongs to.
                Levels: [knowledge, comprehension, application, analysis, synthesis, evaluation]
                query : {query}""")

    reply = response.candidates[0].content.parts[0].text.lower()

    while reply not in ['knowledge', 'comprehension', 'application', 'analysis', 'synthesis', 'evaluation']:
        response = google_client.models.generate_content(
            model="gemini-2.5-flash", 
            contents=
                f"""Extract the blooms level from previous reponse. Answer only in one word without punctuation from: 
                    [knowledge , comprehension , application , analysis, synthesis , evaluation]
                    previous response : {reply}""")

        reply = response.candidates[0].content.parts[0].text.lower()
        time.sleep(10 * random.randint(1,3))

    zsg_pred_labels.append(reply.lower())

  1%|          | 1/126 [00:16<35:17, 16.94s/it]

In [ ]:
print(classification_report(label , [label_mapper[key.lower()] for key in zsg_pred_labels]))

: 

# FEW-Shot

## SONAR

In [ ]:
pred_labels= []

for query in tqdm(queries):
    payload = {
        "model": "sonar",
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": f"""Given the query below, classify which Bloom's Taxonomy level it belongs to.
                    Levels: [knowledge, comprehension, application, analysis, synthesis, evaluation]

                    Examples:

                    1. What is the capital of France? → Knowledge
                    2. Use Ohm’s law to calculate the current in a circuit. → Application
                    3. Critique the author’s argument in the article. → Evaluation
                    4. Summarize the main idea of the passage. → Comprehension             
                    5. Examine the causes of World War I. → Analysis                
                    6. Assess the validity of the research study’s conclusions. → Evaluation
                    7. Propose a plan to reduce plastic pollution in cities. → Synthesis
                    8. Define photosynthesis. → Knowledge
                    9. Apply the concept of supply and demand to predict price changes. → Application              
                    10. Create a new ending for the story. → Synthesis
                    11. Explain in your own words what Newton’s First Law means. → Comprehension
                    12. Identify the relationship between exercise and mental health in the study. → Analysis
                    13. Design an experiment to test the effect of sunlight on plant growth. → Synthesis
                    14. Differentiate between mitosis and meiosis. → Analysis
                    15. Solve the quadratic equation x² – 5x + 6 = 0. → Application
                    16. Judge whether the government’s policy on climate change is effective. → Evaluation
                    17. List the three states of matter. → Knowledge
                    18. Interpret the graph showing population growth. → Comprehension

                    Now classify the question into one Bloom’s Taxonomy level:

                    Question: {query}
                    """}
        ],
        "max_tokens": 100,
        "temperature": 0.5
    }

    response = requests.post(url, headers=headers, json=payload).json()
    reply = response["choices"][0]["message"]["content"]

    while reply not in ['knowledge', 'comprehension', 'application', 'analysis', 'synthesis', 'evaluation']:
        print(reply)

        payload = {
            "model": "sonar",
            "messages": [
                {"role": "system", "content": "You are a helpful assistant."},
                {"role": "user", "content": f"""Extract the blooms level from previous reponse. Answer only in one word without punctuation from: 
                    [knowledge, comprehension, application, analysis, synthesis, evaluation]
                previous response : {reply}"""}
            ],
            "max_tokens": 100,
            "temperature": 0.5
        }
        response = requests.post(url, headers=headers, json=payload).json()
        reply = response["choices"][0]["message"]["content"]

    pred_labels.append(reply.lower())

In [ ]:
print(classification_report(label , [label_mapper[key.lower()] for key in pred_labels]))

## GPT-OSS-120B

In [ ]:
fso_pred_labels = []

for query in tqdm(queries):
    chat_completion = groq_client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": f"""Given the query below, classify which Bloom's Taxonomy level it belongs to.
                    Levels: [knowledge, comprehension, application, analysis, synthesis, evaluation]

                    Examples:

                    1. What is the capital of France? → Knowledge
                    2. Use Ohm’s law to calculate the current in a circuit. → Application
                    3. Critique the author’s argument in the article. → Evaluation
                    4. Summarize the main idea of the passage. → Comprehension             
                    5. Examine the causes of World War I. → Analysis                
                    6. Assess the validity of the research study’s conclusions. → Evaluation
                    7. Propose a plan to reduce plastic pollution in cities. → Synthesis
                    8. Define photosynthesis. → Knowledge
                    9. Apply the concept of supply and demand to predict price changes. → Application              
                    10. Create a new ending for the story. → Synthesis
                    11. Explain in your own words what Newton’s First Law means. → Comprehension
                    12. Identify the relationship between exercise and mental health in the study. → Analysis
                    13. Design an experiment to test the effect of sunlight on plant growth. → Synthesis
                    14. Differentiate between mitosis and meiosis. → Analysis
                    15. Solve the quadratic equation x² – 5x + 6 = 0. → Application
                    16. Judge whether the government’s policy on climate change is effective. → Evaluation
                    17. List the three states of matter. → Knowledge
                    18. Interpret the graph showing population growth. → Comprehension

                    Now classify the question into one Bloom’s Taxonomy level:

                    Question: {query}
                    """,
            }
        ],
        model="openai/gpt-oss-120b",
    )

    reply = chat_completion.choices[0].message.content.lower()

    while reply not in ['knowledge', 'comprehension', 'application', 'analysis', 'synthesis', 'evaluation']:
        chat_completion = groq_client.chat.completions.create(
            messages=[
                {
                    "role": "user",
                    "content": f"""Extract the blooms level from previous reponse. Answer only in one word without punctuation from: 
                        [knowledge , comprehension , application , analysis, synthesis , evaluation]
                    previous response : {reply}""",
                }
            ],
            model="openai/gpt-oss-20b",
        )

        reply = chat_completion.choices[0].message.content.lower()

    fso_pred_labels.append(reply.lower())

In [13]:
print(classification_report(label , [label_mapper[key.lower()] for key in fso_pred_labels]))

              precision    recall  f1-score   support

           0       0.81      0.95      0.88        22
           1       0.82      0.70      0.76        20
           2       0.50      0.47      0.48        15
           3       0.80      0.63      0.71        19
           4       0.68      0.90      0.78        29
           5       1.00      0.76      0.86        21

    accuracy                           0.76       126
   macro avg       0.77      0.74      0.74       126
weighted avg       0.78      0.76      0.76       126



## LLAMA4-Scout

In [14]:
fsl_pred_labels = []

for query in tqdm(queries):
    chat_completion = groq_client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": f"""Classify question based on Bloom’s Taxonomy level using ONLY one word from:
                    [Knowledge, Comprehension, Application, Analysis, Synthesis, Evaluation]

                    Examples:

                    1. What is the capital of France? → Knowledge
                    2. Use Ohm’s law to calculate the current in a circuit. → Application
                    3. Critique the author’s argument in the article. → Evaluation
                    4. Summarize the main idea of the passage. → Comprehension             
                    5. Examine the causes of World War I. → Analysis                
                    6. Assess the validity of the research study’s conclusions. → Evaluation
                    7. Propose a plan to reduce plastic pollution in cities. → Synthesis
                    8. Define photosynthesis. → Knowledge
                    9. Apply the concept of supply and demand to predict price changes. → Application              
                    10. Create a new ending for the story. → Synthesis
                    11. Explain in your own words what Newton’s First Law means. → Comprehension
                    12. Identify the relationship between exercise and mental health in the study. → Analysis
                    13. Design an experiment to test the effect of sunlight on plant growth. → Synthesis
                    14. Differentiate between mitosis and meiosis. → Analysis
                    15. Solve the quadratic equation x² – 5x + 6 = 0. → Application
                    16. Judge whether the government’s policy on climate change is effective. → Evaluation
                    17. List the three states of matter. → Knowledge
                    18. Interpret the graph showing population growth. → Comprehension

                    Now classify the question into one Bloom’s Taxonomy level:

                    Question: {query}
                    """,
            }
        ],
        model="meta-llama/llama-4-scout-17b-16e-instruct",
    )

    reply = chat_completion.choices[0].message.content.lower()

    while reply not in ['knowledge', 'comprehension', 'application', 'analysis', 'synthesis', 'evaluation']:
        chat_completion = groq_client.chat.completions.create(
            messages=[
                {
                    "role": "user",
                    "content": f"""Extract the blooms level from previous reponse. Answer only in one word without punctuation from: 
                        [knowledge , comprehension , application , analysis, synthesis , evaluation]
                    previous response : {reply}""",
                }
            ],
            model="meta-llama/llama-4-scout-17b-16e-instruct",
        )

        reply = chat_completion.choices[0].message.content.lower()

    fsl_pred_labels.append(reply.lower())

100%|██████████| 126/126 [03:50<00:00,  1.83s/it]


In [15]:
print(classification_report(label , [label_mapper[key.lower()] for key in fsl_pred_labels]))

              precision    recall  f1-score   support

           0       0.87      0.91      0.89        22
           1       0.79      0.75      0.77        20
           2       0.50      0.20      0.29        15
           3       0.87      0.68      0.76        19
           4       0.61      0.97      0.75        29
           5       0.88      0.71      0.79        21

    accuracy                           0.75       126
   macro avg       0.75      0.70      0.71       126
weighted avg       0.75      0.75      0.73       126



## GEMINI

In [ ]:
fsg_pred_labels = []

for query in tqdm(queries):
    chat_completion = groq_client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": f"""Classify question based on Bloom’s Taxonomy level using ONLY one word from:
                    [Knowledge, Comprehension, Application, Analysis, Synthesis, Evaluation]

                    Examples:

                    1. What is the capital of France? → Knowledge
                    2. Use Ohm’s law to calculate the current in a circuit. → Application
                    3. Critique the author’s argument in the article. → Evaluation
                    4. Summarize the main idea of the passage. → Comprehension             
                    5. Examine the causes of World War I. → Analysis                
                    6. Assess the validity of the research study’s conclusions. → Evaluation
                    7. Propose a plan to reduce plastic pollution in cities. → Synthesis
                    8. Define photosynthesis. → Knowledge
                    9. Apply the concept of supply and demand to predict price changes. → Application              
                    10. Create a new ending for the story. → Synthesis
                    11. Explain in your own words what Newton’s First Law means. → Comprehension
                    12. Identify the relationship between exercise and mental health in the study. → Analysis
                    13. Design an experiment to test the effect of sunlight on plant growth. → Synthesis
                    14. Differentiate between mitosis and meiosis. → Analysis
                    15. Solve the quadratic equation x² – 5x + 6 = 0. → Application
                    16. Judge whether the government’s policy on climate change is effective. → Evaluation
                    17. List the three states of matter. → Knowledge
                    18. Interpret the graph showing population growth. → Comprehension

                    Now classify the question into one Bloom’s Taxonomy level:

                    Question: {query}
                    """,
            }
        ],
        model="meta-llama/llama-4-scout-17b-16e-instruct",
    )

    reply = chat_completion.choices[0].message.content.lower()

    while reply not in ['knowledge', 'comprehension', 'application', 'analysis', 'synthesis', 'evaluation']:
        chat_completion = groq_client.chat.completions.create(
            messages=[
                {
                    "role": "user",
                    "content": f"""Extract the blooms level from previous reponse. Answer only in one word without punctuation from: 
                        [knowledge , comprehension , application , analysis, synthesis , evaluation]
                    previous response : {reply}""",
                }
            ],
            model="meta-llama/llama-4-scout-17b-16e-instruct",
        )

        reply = chat_completion.choices[0].message.content.lower()

    fsg_pred_labels.append(reply.lower())

In [ ]:
print(classification_report(label , [label_mapper[key.lower()] for key in fsl_pred_labels]))

# Chain-of-Thought

## GPT-OSS-120B

In [21]:
coto_pred_labels = []

for query in tqdm(queries):
    # Reason

    chat_completion = groq_client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": f"""
                    Given the query below, reason about which Bloom's Taxonomy level it belongs to.
                    Levels: [knowledge, comprehension, application, analysis, synthesis, evaluation]

                    Query: {query}""",
            }
        ],
        model="openai/gpt-oss-120b",
    )

    reply = chat_completion.choices[0].message.content.lower()

    # Summarize and Classify
    
    chat_completion = groq_client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": f"""
                    Summarize the reasoning into exactly one Bloom's Taxonomy level.
                    Levels: [knowledge, comprehension, application, analysis, synthesis, evaluation]

                    Reasoning: {reply}

                    Answer in one word ONLY""",
            }
        ],
        model="openai/gpt-oss-120b",
    )

    reply = chat_completion.choices[0].message.content.lower()

    while reply not in ['knowledge', 'comprehension', 'application', 'analysis', 'synthesis', 'evaluation']:
        chat_completion = groq_client.chat.completions.create(
            messages=[
                {
                    "role": "user",
                    "content": f"""Extract the blooms level from previous reponse. Answer only in one word without punctuation from: 
                        [knowledge , comprehension , application , analysis, synthesis , evaluation]
                    previous response : {reply}""",
                }
            ],
            model="openai/gpt-oss-20b",
        )

        reply = chat_completion.choices[0].message.content.lower()
    time.sleep(5 * random.randint(0, 2))

    coto_pred_labels.append(reply.lower())

100%|██████████| 126/126 [12:00<00:00,  5.72s/it]


In [22]:
print(classification_report(label , [label_mapper[key.lower()] for key in coto_pred_labels]))

              precision    recall  f1-score   support

           0       0.87      0.91      0.89        22
           1       1.00      0.65      0.79        20
           2       0.45      0.33      0.38        15
           3       0.72      0.68      0.70        19
           4       0.63      0.90      0.74        29
           5       0.90      0.86      0.88        21

    accuracy                           0.75       126
   macro avg       0.76      0.72      0.73       126
weighted avg       0.77      0.75      0.75       126



## LLAMA4-Scout

In [24]:
cotl_pred_labels = []

for query in tqdm(queries):
    # Reason

    chat_completion = groq_client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": f"""
                    Given the query below, reason about which Bloom's Taxonomy level it belongs to.
                    Levels: [knowledge, comprehension, application, analysis, synthesis, evaluation]

                    Query: {query}""",
            }
        ],
        model="deepseek-r1-distill-llama-70b",
    )

    reply = chat_completion.choices[0].message.content.lower()

    # Summarize and Classify
    
    chat_completion = groq_client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": f"""
                    Summarize the reasoning into exactly one Bloom's Taxonomy level.
                    Levels: [knowledge, comprehension, application, analysis, synthesis, evaluation]

                    Reasoning: {reply}

                    Answer in one word ONLY""",
            }
        ],
        model="meta-llama/llama-4-scout-17b-16e-instruct",
    )

    reply = chat_completion.choices[0].message.content.lower()

    while reply not in ['knowledge', 'comprehension', 'application', 'analysis', 'synthesis', 'evaluation']:
        chat_completion = groq_client.chat.completions.create(
            messages=[
                {
                    "role": "user",
                    "content": f"""Extract the blooms level from previous reponse. Answer only in one word without punctuation from: 
                        [knowledge , comprehension , application , analysis, synthesis , evaluation]
                    previous response : {reply}""",
                }
            ],
            model="meta-llama/llama-4-scout-17b-16e-instruct",
        )

        reply = chat_completion.choices[0].message.content.lower()
    time.sleep(5 * random.randint(0, 2))

    cotl_pred_labels.append(reply.lower())

100%|██████████| 126/126 [15:16<00:00,  7.27s/it]


In [25]:
print(classification_report(label , [label_mapper[key.lower()] for key in cotl_pred_labels]))

              precision    recall  f1-score   support

           0       0.88      0.95      0.91        22
           1       0.92      0.60      0.73        20
           2       0.38      0.33      0.36        15
           3       0.76      0.68      0.72        19
           4       0.65      0.90      0.75        29
           5       0.89      0.81      0.85        21

    accuracy                           0.75       126
   macro avg       0.75      0.71      0.72       126
weighted avg       0.76      0.75      0.74       126



# Save Labels

In [ ]:
label_data = {
    'zero_shot_gpt' : zso_pred_labels,
    'zero_shot_llama' : zsl_pred_labels,
    'few_shot_gpt' : fso_pred_labels, 
    'few_shot_llama' : fsl_pred_labels
            }

df = pd.DataFrame(data= label_data)

df.to_csv('context_groq.csv', index=False) 

In [26]:
label_data = {
    'cot_gpt' : coto_pred_labels,
    'cot_llama' : cotl_pred_labels
            }

df = pd.DataFrame(data= label_data)

df.to_csv('context_cot.csv', index=False) 